# Problem Set 4: What Do Presidents Tell Congress?

QSS 20 · Modern Statistical Computing · Fall 2026

**Out:** Thursday, October 22 · **Due:** Thursday, October 29, 11:59pm

## Overview

The Constitution asks the president to give Congress "Information of the State
of the Union" from time to time. Every president since Washington has done it
about once a year: 236 addresses from 1790 to 2026, by 43 presidents. For more
than a century the address was a written report that a clerk read aloud.
Woodrow Wilson revived the spoken address in 1913. Since Franklin Roosevelt it
has usually been a broadcast speech to a joint session of Congress.

In this problem set you will build a corpus table from one large text file, pull words
out of the prose with regular expressions, trace words across two centuries,
turn the modern addresses into a matrix, compare the addresses, and let a topic
model say what the corpus is about. 

**Submission.** Work in `ps4.ipynb`. Each task has an empty cell under it and
a page break after it. Keep that layout, so page tagging on Gradescope stays
simple. Run Kernel, then Restart Kernel and Run All Cells. Export with
`quarto render ps4.ipynb --to typst`. Upload `ps4.ipynb` and `ps4.pdf` to
Gradescope. The Problem Set Reflections at the end are part of the set.

**Data.** `data/stateoftheunion1790-2026.txt` holds every address in one
file. A table of contents comes first; then the addresses follow, separated by
a line of three asterisks. Each address starts with three lines, the title, the
president, and the date, then a blank line, then the text. See the Appendix for
where the file came from. `data/presidents.csv` maps each president's name, as
it is written in the file, to a party.

Sections 1 and 2 use all 236 addresses. Sections 3 to 5 use the addresses from
**1934 onward**, the era of the spoken, broadcast address and of the modern
Democratic and Republican parties. Before 1934 the file mixes written reports
with speeches, and the party labels name organizations that no longer map onto
the two-party contest.

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CORPUS = Path("data/stateoftheunion1790-2026.txt")
PARTIES = Path("data/presidents.csv")

{{< pagebreak >}}

## 1. Build the corpus table (18 points)

Nothing in this section is analysis. It is the work that has to happen before
analysis is possible, and it is where most text projects lose their time.

**1.1 (8 points).** Read the file as one string. Split it into records at the
separator line (the three asterisks sit on a line of their own), and drop the
first piece, the table of contents. Put the records in a Series, strip the
whitespace around each one, and split each record at its first three line
breaks: `.str.split("\n", n=3, expand=True)` splits at most `n` times and
returns one column per piece. Rename the four columns `title`, `president`,
`date`, and `text`, then strip the whitespace from every column. Two president
names in the file end with trailing spaces, and they must match the party
table later. Extract the four-digit `year` from the date with a regular
expression. Report the number of rows, the first and last year, and the number
of distinct presidents.

{{< pagebreak >}}

**1.2 (4 points).** Merge the party table onto the corpus with a validated
many-to-one merge (many addresses, one row per president). Report the rows
per party and the count of unmatched rows.

{{< pagebreak >}}

**1.3 (6 points).** Build a `tokens` column: replace the curly apostrophe `’`
with the straight one `'`, lowercase, then find every run of letters and
apostrophes with `.str.findall`. Store each address's length in tokens. Plot
length against year with both axes labeled. Describe the pattern across the
two centuries in two sentences, and name the shortest and the longest address.
Print the last 300 characters of the shortest one: is it a complete address?

{{< pagebreak >}}

## 2. Read the text with regular expressions (20 points)

Section 1 split a tidy header. Prose is not tidy. This section asks the same
tool to work on the addresses themselves.

**2.1 (6 points).** Presidents say "our" constantly, and the word after it
says who they think "we" are and what "we" own. Use `re.findall` with one
capture group on the lowercased file to pull every word that follows `our`
across all 236 addresses. The file wraps its lines, so match the gap after
`our` with `\s+`, not a space. Put the matches in a Series and report the 10 most
common. Write one sentence on what the list says about the subject of the
address.

{{< pagebreak >}}

**2.2 (8 points).** Count the mentions of three things that history books
attach to particular decades: `war` (and `wars`), `soviet` (and `soviets`),
and `terror` with the words built on it (`terrorism`, `terrorist`,
`terrorists`). Write one pattern for each. Put word boundaries on both sides
of `war` and `soviet`; `terror` takes only the opening one, because `\w*`
runs to the end of the word. Count the matches in each address with `.str.findall` and `.str.len`, and
divide by the address's length to get a rate per 1,000 tokens. Add a `decade`
column: `(corpus["year"] / 10).astype(int) * 10` rounds each year down to its
decade. Average the three rates by decade and plot the three series on one
labeled figure with a legend. Name the decade where each word peaks. Then
explain in one sentence why the word boundary matters for `war`, naming a
word that would be miscounted without it.

{{< pagebreak >}}

**2.3 (6 points).** For every address, compute the rate per 1,000 tokens of
the words `treasury` and `jobs`, with word boundaries on both sides. Plot both
rates against year on one labeled figure with a legend. Create an `era`
column that splits at 1934 and report the mean rate of each word by era.
Describe what happened to the two words in two sentences. Add one sentence on
why a **rate** rather than a raw count is the honest measure here, using your
Section 1.3 figure as the evidence.

{{< pagebreak >}}

## 3. From text to matrix (18 points)

A statistical method cannot read prose. It needs a table of numbers with one
row per document and one column per term. Building that table forces
decisions that change every result downstream, and this section makes two of
them visible. From here on, use the addresses from **1934 onward**. Keep them
in a new DataFrame with a fresh index, and report how many addresses and how
many of each party it holds.

**3.1 (5 points).** Build a document-term matrix of raw counts with
`CountVectorizer(stop_words="english", min_df=2)`. Report its shape and the
share of cells that are nonzero.

{{< pagebreak >}}

**3.2 (6 points).** Build a TF-IDF matrix with
`TfidfVectorizer(stop_words="english", min_df=2, max_df=0.8)`. The new
argument `max_df=0.8` drops every term that appears in more than 80 percent
of the addresses. List the terms it dropped: turn the two vocabularies into Python sets, so
that `set(vocab) - set(terms)` keeps the words in the first and not the
second; sort the difference and print the count and the first 40 words.
Explain in one sentence why a word that appears in almost every address
cannot help to tell the addresses apart.

{{< pagebreak >}}

**3.3 (7 points).** Take the **2026** address. Report its 8 highest-scoring
terms by raw count and by TF-IDF weight side by side. Explain in two
sentences why the two lists differ. Then look down the TF-IDF list: which
terms are unlikely to have been spoken in the speech? Check how they are
used in the raw text before you answer. `re.findall` with up to 40
characters of context on each side shows every use of a word, for example
`re.findall(r".{0,40}laughter.{0,40}", text, flags=re.IGNORECASE)` on the
2026 text. Say where each of those terms came from.

{{< pagebreak >}}

## 4. Compare documents (8 points)

**4.1 (8 points).** Compute the cosine similarity between every pair of
addresses from 1934 on, from the TF-IDF matrix. Build a DataFrame with the
year, president, party, and similarity to the **2026** address, sort it from
most to least similar, and show the first ten rows. The first row is the
address itself. Write one sentence on what the ten rows have in common.

{{< pagebreak >}}

## 5. What the topics are about (26 points)

Sections 3 and 4 compared whole documents. A topic model works one level
down. It looks for groups of words that rise and fall together across the
corpus, then describes each address as a mixture of those groups. Nobody
tells it what the groups should be.

**5.1 (6 points).** Sections 3 and 4 kept `CountVectorizer`'s default token
rule, which splits text at every character that is not a letter or digit.
An apostrophe or a hyphen therefore cuts a word in two, and fragments such
as `ve` (from `we've`), `ll` (from `we'll`), and `al` (from `al-Qaida`) sit
in the vocabulary. For the topic model, build a new TF-IDF matrix with
`TfidfVectorizer(stop_words="english", min_df=2, max_df=0.8,
token_pattern=r"\b[a-z]{5,}\b")`: the same `min_df` and `max_df` as
Section 3 plus a token rule that keeps only runs of five or more letters.
Fit `NMF` with `n_components=6` on it, using `init="nndsvda"`,
`max_iter=600`, and `random_state=42`. Print the 8 highest-weight terms in
each topic and give each topic a short name of your own in plain English.

{{< pagebreak >}}

**5.2 (5 points).** Plot the fifteen strongest words of each topic as six
horizontal bar panels in one figure, two rows of three, as the lecture did
for the Federalist essays, with the strongest word at the top of each panel
and the topic number as the panel title.

{{< pagebreak >}}

**5.3 (5 points).** Assign each address to its highest-weight topic. Report
the first year, last year, and count of each topic's addresses, and crosstab
topic against party.

{{< pagebreak >}}

**5.4 (5 points).** Turn each address's four weights into shares that sum to
1, and plot the shares as one stacked bar per address in the order of the
years, as the lecture did by essay number. Label the x axis with every tenth
year.

{{< pagebreak >}}

**5.5 (5 points).** Sum the count matrix of 3.1 over the Democratic addresses
and over the Republican addresses, so that each term has one count per party.
Keep the terms used at least 40 times in total and at least once by each
party, and drop any term that contains a digit (the vocabulary includes year
numbers such as `1946`). For each term, compute the ratio of its Democratic
rate (count divided by the party's total count) to its Republican rate.
Report the six most Republican-leaning and the six most Democratic-leaning
terms with their ratios. Then look at the two most lopsided terms, the one
with the highest ratio and the one with the lowest: why do you think those
two are the extremes? Check the raw text before you answer.

{{< pagebreak >}}

## Problem Set Reflections (10 points)

Every problem set ends with these reflections. Answer them honestly. You are
graded on the specificity and honesty in the reflections you show, not on how
much AI you used or did not use. You cannot use AI for this part, and we are
not checking for spelling or grammar. Write as much as each answer needs.

Write your answer under each heading.

### AI use

List each substantive use of an AI assistant: the tool and model, what you
asked for, what it gave back, how you checked it, and whether you kept or
rejected the result. Write "None" if you used none.

*Your answer:*

### People consulted

Who you talked to about this assignment, and what about.

*Your answer:*

### Where you got stuck

The step that blocked you, and what got you unstuck.

*Your answer:*

### What you understand now

One thing you understand now that you did not when you started.

*Your answer:*

### Hours spent

An honest estimate of the time this took.

*Your answer:*

{{< pagebreak >}}

## Appendix: the corpus and the labels

The file is the Project Gutenberg collection *Complete State of the Union
Addresses*, extended by the course: addresses from 2002 on come from the
American Presidency Project at UC Santa Barbara.

Five of the 236 entries carry another title. A president's first address to a
joint session of Congress, in the year after an election, is not formally a
State of the Union, and the file calls it an "Address on Administration
Goals" or an "Address Before a Joint Session of Congress". The file counts
them, and so does this problem set. The transcripts of 2025 and 2026 keep
stage directions such as "[Applause]" and speaker labels such as "Audience
members."; Sections 3 and 5 meet the consequences. One entry is cut short in
the file, and Section 1.3 asks you to find it.

`presidents.csv` gives each president's party from the standard historical
record. Three cases are worth knowing. Washington belonged to no party and is
labeled Unaffiliated. John Tyler is labeled Whig, the party that elected him,
though it expelled him in 1841. Andrew Johnson is labeled Democratic, though
he was elected on the National Union ticket with Lincoln. 